In [1]:
!pwd

/Users/user/Downloads/MTECH2526/EBA5006_project/bixi-analytics


## processing of the files

### trip data db

In [3]:
import pathlib
import pandas as pd

p = pathlib.Path('../data')
csv_files = sorted(p.rglob('*.csv'))

if not csv_files:
    print('No CSV files found under data/')
else:
    for f in csv_files:
        print('FILE:', f)
        try:
            df = pd.read_csv(f, nrows=10)
            print('  HEADER:', df.columns.tolist())
            print('  Shape (first 10 rows):', df.shape)
            print(df.to_string())
        except Exception as e:
            print('  ERROR reading file:', e)
        print()

FILE: ../data/Cleaned_Station_Dimension.csv
  HEADER: ['station_name_standardized', 'latitude', 'longitude', 'elevation_meters', 'nearest_metro_dist_meters']
  Shape (first 10 rows): (10, 5)
               station_name_standardized   latitude  longitude  elevation_meters  nearest_metro_dist_meters
0                    AYLMER / SHERBROOKE  45.506295 -73.572827              43.0                     264.45
1                   DE ROUEN / LESPRANCE  45.538859 -73.552818              25.0                     337.29
2                 STE-FAMILLE / DES PINS  45.512856 -73.576928              50.0                     859.21
3                     WILLIAM / ST-HENRI  45.498887 -73.557117              19.0                     587.43
4  PARC DE ROXBORO (11E AVENUE / 9E RUE)  45.505975 -73.803556               NaN                     774.99
5                   BOILEAU / LACORDAIRE  45.572558 -73.541075               NaN                     563.78
6           HLÈNE-BAILLARGEON / ST-DENIS  45.529740 -

In [4]:
# Read headers from all files
headers = {}
for f in csv_files:
    try:
        df_header = pd.read_csv(f, nrows=1)
        headers[str(f)] = list(df_header.columns)
    except Exception as e:
        print(f"Error reading {f}: {e}")

# Check if all headers are the same
print("Headers per file:")
for file, header in headers.items():
    print(f"  {file}: {header}")

# Check if all headers match
all_same = len(set(tuple(h) for h in headers.values())) == 1

if all_same:
    print("\n✓ All headers are identical!")
    print("\nConcatenating top 10 rows from all files...")
    
    dfs = []
    for f in csv_files:
        try:
            df = pd.read_csv(f, nrows=10)
            dfs.append(df)
        except Exception as e:
            print(f"Error reading {f}: {e}")
    
    combined_df = pd.concat(dfs, ignore_index=True)
    print(f"\nCombined shape: {combined_df.shape}")
    print("\nCombined data:")
    print(combined_df)
else:
    print("\n✗ Headers are different across files!")
    print("Cannot concatenate data.")


## combine the dataframe
# # French to English header mapping
french_to_english = {
    'STARTSTATIONNAME': 'start_station_name',
    'STARTSTATIONARRONDISSEMENT': 'start_station_district',
    'STARTSTATIONLATITUDE': 'start_station_latitude',
    'STARTSTATIONLONGITUDE': 'start_station_longitude',
    'ENDSTATIONNAME': 'end_station_name',
    'ENDSTATIONARRONDISSEMENT': 'end_station_district',
    'ENDSTATIONLATITUDE': 'end_station_latitude',
    'ENDSTATIONLONGITUDE': 'end_station_longitude',
    'STARTTIMEMS': 'start_time_ms',
    'ENDTIMEMS': 'end_time_ms',
}

print("French to English Header Mapping:")
for fr, en in french_to_english.items():
    print(f"  {fr} → {en}")

# Apply mapping to combined_df
if 'combined_df' in dir():
    # Only rename columns that exist in the dataframe
    rename_mapping = {col: french_to_english.get(col, col) for col in combined_df.columns}
    combined_df_en = combined_df.rename(columns=rename_mapping)
    
    print("\nUpdated headers:")
    print(combined_df_en.columns.tolist())
    print("\nData with English headers:")
    print(combined_df_en)    

Headers per file:
  ../data/Cleaned_Station_Dimension.csv: ['station_name_standardized', 'latitude', 'longitude', 'elevation_meters', 'nearest_metro_dist_meters']
  ../data/DonneesOuvertes (2).csv: ['STARTSTATIONNAME', 'STARTSTATIONARRONDISSEMENT', 'STARTSTATIONLATITUDE', 'STARTSTATIONLONGITUDE', 'ENDSTATIONNAME', 'ENDSTATIONARRONDISSEMENT', 'ENDSTATIONLATITUDE', 'ENDSTATIONLONGITUDE', 'STARTTIMEMS', 'ENDTIMEMS']
  ../data/DonneesOuvertes2025_010203040506070809101112.csv: ['STARTSTATIONNAME', 'STARTSTATIONARRONDISSEMENT', 'STARTSTATIONLATITUDE', 'STARTSTATIONLONGITUDE', 'ENDSTATIONNAME', 'ENDSTATIONARRONDISSEMENT', 'ENDSTATIONLATITUDE', 'ENDSTATIONLONGITUDE', 'STARTTIMEMS', 'ENDTIMEMS']

✗ Headers are different across files!
Cannot concatenate data.
French to English Header Mapping:
  STARTSTATIONNAME → start_station_name
  STARTSTATIONARRONDISSEMENT → start_station_district
  STARTSTATIONLATITUDE → start_station_latitude
  STARTSTATIONLONGITUDE → start_station_longitude
  ENDSTATIONNA

In [6]:
# load the 3 csv files loaded into 3 separate dataframes and then concatenate them into a single dataframe.
dfs = []
for f in csv_files:
    try:
        df = pd.read_csv(f)
        dfs.append(df)
    except Exception as e:
        print(f"Error reading {f}: {e}")
# print first 10 rows of the first csv file dataframe
if dfs:
    print("First 10 rows of the first dataframe:")
    print(dfs[0].head(10))

First 10 rows of the first dataframe:
               station_name_standardized   latitude  longitude  \
0                    AYLMER / SHERBROOKE  45.506295 -73.572827   
1                   DE ROUEN / LESPRANCE  45.538859 -73.552818   
2                 STE-FAMILLE / DES PINS  45.512856 -73.576928   
3                     WILLIAM / ST-HENRI  45.498887 -73.557117   
4  PARC DE ROXBORO (11E AVENUE / 9E RUE)  45.505975 -73.803556   
5                   BOILEAU / LACORDAIRE  45.572558 -73.541075   
6           HLÈNE-BAILLARGEON / ST-DENIS  45.529740 -73.595270   
7            JEANNE-MANCE / REN-LÉVESQUE  45.506371 -73.564201   
8            DU MARCH CENTRAL / CRÉMAZIE  45.533417 -73.648884   
9                   DE GASP / ST-VIATEUR  45.527792 -73.597608   

   elevation_meters  nearest_metro_dist_meters  
0              43.0                     264.45  
1              25.0                     337.29  
2              50.0                     859.21  
3              19.0                    

### load other DB 

In [2]:
import requests
import json
import sqlite3
import csv
from datetime import datetime
import os

# Database file name
DB_FILE = "bikeshare.db"
# Directory for CSV exports (create if not exists)
CSV_EXPORT_DIR = "csv_exports"

# List of feeds to process
FEEDS = [
    {"name": "station_information", "url": "https://gbfs.velobixi.com/gbfs/2-2/en/station_information.json"},
    {"name": "system_alerts", "url": "https://gbfs.velobixi.com/gbfs/2-2/en/system_alerts.json"},
    {"name": "system_information", "url": "https://gbfs.velobixi.com/gbfs/2-2/en/system_information.json"},
    {"name": "vehicle_types", "url": "https://gbfs.velobixi.com/gbfs/2-2/en/vehicle_types.json"},
    {"name": "station_status", "url": "https://gbfs.velobixi.com/gbfs/2-2/en/station_status.json"}
]

def create_tables(conn):
    """Create all necessary tables if they don't exist"""
    cursor = conn.cursor()
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS stations (
            station_id TEXT PRIMARY KEY,
            external_id TEXT,
            name TEXT,
            short_name TEXT,
            lat REAL,
            lon REAL,
            rental_methods TEXT,
            capacity INTEGER,
            electric_bike_surcharge_waiver INTEGER,
            is_charging INTEGER,
            eightd_has_key_dispenser INTEGER,
            has_kiosk INTEGER,
            last_updated INTEGER
        )
    ''')
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS system_alerts (
            alert_id TEXT PRIMARY KEY,
            type TEXT,
            times TEXT,
            url TEXT,
            summary TEXT,
            description TEXT,
            alert_last_updated INTEGER,
            feed_last_updated INTEGER
        )
    ''')
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS system_information (
            system_id TEXT PRIMARY KEY,
            language TEXT,
            name TEXT,
            short_name TEXT,
            operator TEXT,
            url TEXT,
            purchase_url TEXT,
            start_date TEXT,
            phone_number TEXT,
            email TEXT,
            timezone TEXT,
            license_url TEXT,
            last_updated INTEGER
        )
    ''')
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS vehicle_types (
            vehicle_type_id TEXT PRIMARY KEY,
            name TEXT,
            form_factor TEXT,
            propulsion_type TEXT,
            max_range_meters INTEGER,
            name_length TEXT,
            last_updated INTEGER
        )
    ''')
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS station_status (
            station_id TEXT,
            num_bikes_available INTEGER,
            num_ebikes_available INTEGER,
            num_bikes_disabled INTEGER,
            num_docks_available INTEGER,
            num_docks_disabled INTEGER,
            is_installed INTEGER,
            is_renting INTEGER,
            is_returning INTEGER,
            last_reported INTEGER,
            vehicle_types_available TEXT,
            vehicle_docks_available TEXT,
            feed_last_updated INTEGER,
            PRIMARY KEY (station_id, feed_last_updated)
        )
    ''')
    
    conn.commit()

def export_table_to_csv(conn, table_name):
    """Export a single table to a CSV file"""
    cursor = conn.cursor()
    
    # Get column names
    cursor.execute(f"PRAGMA table_info({table_name})")
    columns = [col[1] for col in cursor.fetchall()]
    
    # Fetch all rows
    cursor.execute(f"SELECT * FROM {table_name}")
    rows = cursor.fetchall()
    
    if not rows:
        print(f"  ⚠️  Table '{table_name}' is empty, skipping CSV export.")
        return
    
    # Create export directory if needed
    os.makedirs(CSV_EXPORT_DIR, exist_ok=True)
    
    # Write CSV file
    csv_file = os.path.join(CSV_EXPORT_DIR, f"{table_name}.csv")
    with open(csv_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(columns)  # header
        writer.writerows(rows)
    
    print(f"  📄 Exported {len(rows)} rows to {csv_file}")

def export_all_tables_to_csv(conn):
    """Export all tables to CSV files"""
    print(f"\n📁 Exporting tables to CSV in '{CSV_EXPORT_DIR}' directory...")
    tables = [
        "stations", 
        "system_alerts", 
        "system_information", 
        "vehicle_types", 
        "station_status"
    ]
    for table in tables:
        export_table_to_csv(conn, table)

# [Include all the process_* functions here exactly as before]
def process_station_information(data, cursor, last_updated):
    stations = data['data']['stations']
    print(f"  Processing {len(stations)} stations")
    for station in stations:
        rental_methods_json = json.dumps(station.get('rental_methods', []))
        cursor.execute('''
            INSERT OR REPLACE INTO stations (
                station_id, external_id, name, short_name, lat, lon,
                rental_methods, capacity, electric_bike_surcharge_waiver,
                is_charging, eightd_has_key_dispenser, has_kiosk, last_updated
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            station['station_id'],
            station.get('external_id'),
            station['name'],
            station.get('short_name'),
            station['lat'],
            station['lon'],
            rental_methods_json,
            station['capacity'],
            1 if station.get('electric_bike_surcharge_waiver', False) else 0,
            1 if station.get('is_charging', False) else 0,
            1 if station.get('eightd_has_key_dispenser', False) else 0,
            1 if station.get('has_kiosk', False) else 0,
            last_updated
        ))

def process_system_alerts(data, cursor, last_updated):
    alerts = data['data'].get('alerts', [])
    print(f"  Processing {len(alerts)} alerts")
    for alert in alerts:
        times_json = json.dumps(alert.get('times', []))
        cursor.execute('''
            INSERT OR REPLACE INTO system_alerts (
                alert_id, type, times, url, summary, description, 
                alert_last_updated, feed_last_updated
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            alert['alert_id'],
            alert.get('type'),
            times_json,
            alert.get('url'),
            alert.get('summary'),
            alert.get('description'),
            alert.get('last_updated'),
            last_updated
        ))

def process_system_information(data, cursor, last_updated):
    sys_info = data['data']
    print("  Processing system information")
    cursor.execute('''
        INSERT OR REPLACE INTO system_information (
            system_id, language, name, short_name, operator, url,
            purchase_url, start_date, phone_number, email, timezone,
            license_url, last_updated
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (
        sys_info.get('system_id'),
        sys_info.get('language'),
        sys_info.get('name'),
        sys_info.get('short_name'),
        sys_info.get('operator'),
        sys_info.get('url'),
        sys_info.get('purchase_url'),
        sys_info.get('start_date'),
        sys_info.get('phone_number'),
        sys_info.get('email'),
        sys_info.get('timezone'),
        sys_info.get('license_url'),
        last_updated
    ))

def process_vehicle_types(data, cursor, last_updated):
    vehicle_types = data['data'].get('vehicle_types', [])
    print(f"  Processing {len(vehicle_types)} vehicle types")
    for vt in vehicle_types:
        cursor.execute('''
            INSERT OR REPLACE INTO vehicle_types (
                vehicle_type_id, name, form_factor, propulsion_type,
                max_range_meters, name_length, last_updated
            ) VALUES (?, ?, ?, ?, ?, ?, ?)
        ''', (
            vt['vehicle_type_id'],
            vt.get('name'),
            vt.get('form_factor'),
            vt.get('propulsion_type'),
            vt.get('max_range_meters'),
            vt.get('name_length'),
            last_updated
        ))

def process_station_status(data, cursor, last_updated):
    stations = data['data']['stations']
    print(f"  Processing {len(stations)} station statuses")
    for station in stations:
        vehicle_types_json = json.dumps(station.get('vehicle_types_available', []))
        vehicle_docks_json = json.dumps(station.get('vehicle_docks_available', []))
        cursor.execute('''
            INSERT INTO station_status (
                station_id, num_bikes_available, num_ebikes_available,
                num_bikes_disabled, num_docks_available, num_docks_disabled,
                is_installed, is_renting, is_returning, last_reported,
                vehicle_types_available, vehicle_docks_available, feed_last_updated
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            station['station_id'],
            station.get('num_bikes_available', 0),
            station.get('num_ebikes_available', 0),
            station.get('num_bikes_disabled', 0),
            station.get('num_docks_available', 0),
            station.get('num_docks_disabled', 0),
            1 if station.get('is_installed', False) else 0,
            1 if station.get('is_renting', False) else 0,
            1 if station.get('is_returning', False) else 0,
            station.get('last_reported', last_updated),
            vehicle_types_json,
            vehicle_docks_json,
            last_updated
        ))

def main():
    print(f"Starting data fetch at {datetime.now()}")
    
    # Connect to database
    conn = sqlite3.connect(DB_FILE)
    create_tables(conn)
    cursor = conn.cursor()
    
    # Process each feed
    for feed in FEEDS:
        print(f"\n📡 Fetching {feed['name']}...")
        try:
            response = requests.get(feed['url'], timeout=10)
            if response.status_code == 200:
                data = response.json()
                last_updated = data['last_updated']
                
                if feed['name'] == 'station_information':
                    process_station_information(data, cursor, last_updated)
                elif feed['name'] == 'system_alerts':
                    process_system_alerts(data, cursor, last_updated)
                elif feed['name'] == 'system_information':
                    process_system_information(data, cursor, last_updated)
                elif feed['name'] == 'vehicle_types':
                    process_vehicle_types(data, cursor, last_updated)
                elif feed['name'] == 'station_status':
                    process_station_status(data, cursor, last_updated)
                
                conn.commit()
                print(f"  ✅ {feed['name']} saved successfully")
            else:
                print(f"  ❌ Failed: HTTP {response.status_code}")
        except Exception as e:
            print(f"  ❌ Error: {str(e)}")
    
    # Export all tables to CSV
    export_all_tables_to_csv(conn)
    
    # Close connection
    conn.close()
    print(f"\n✅ All feeds processed. Database: {DB_FILE}")
    print(f"📁 CSV files saved in '{CSV_EXPORT_DIR}' folder")
    print(f"Completed at {datetime.now()}")

if __name__ == "__main__":
    main()

Starting data fetch at 2026-03-03 15:20:09.379557

📡 Fetching station_information...
  Processing 240 stations
  ✅ station_information saved successfully

📡 Fetching system_alerts...
  Processing 35 alerts
  ✅ system_alerts saved successfully

📡 Fetching system_information...
  Processing system information
  ✅ system_information saved successfully

📡 Fetching vehicle_types...
  Processing 5 vehicle types
  ✅ vehicle_types saved successfully

📡 Fetching station_status...
  Processing 240 station statuses
  ✅ station_status saved successfully

📁 Exporting tables to CSV in 'csv_exports' directory...
  📄 Exported 240 rows to csv_exports/stations.csv
  📄 Exported 35 rows to csv_exports/system_alerts.csv
  📄 Exported 1 rows to csv_exports/system_information.csv
  📄 Exported 5 rows to csv_exports/vehicle_types.csv
  📄 Exported 240 rows to csv_exports/station_status.csv

✅ All feeds processed. Database: bikeshare.db
📁 CSV files saved in 'csv_exports' folder
Completed at 2026-03-03 15:20:18.36

In [ ]:
## also save the station information to a CSV file for easier access and analysis in tools like Excel or pandas.
import csv
from datetime import datetime

# 1. Fetch the data from the URL
url = "https://gbfs.velobixi.com/gbfs/2-2/en/station_information.json"
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    stations = data['data']['stations']
    last_updated = data['last_updated']  # Unix timestamp
    print(f"Found {len(stations)} stations.\n")

    # 2. Define CSV file name and field names (columns)
    csv_file = "stations.csv"
    fieldnames = [
        "station_id", "external_id", "name", "short_name", "lat", "lon",
        "rental_methods", "capacity", "electric_bike_surcharge_waiver",
        "is_charging", "eightd_has_key_dispenser", "has_kiosk", "last_updated"
    ]

    # 3. Open CSV file for writing
    with open(csv_file, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()  # write column headers

        # 4. Write each station as a row
        for station in stations:
            # Convert rental_methods list to a comma-separated string
            rental_methods_str = ', '.join(station.get('rental_methods', []))

            # Prepare row dictionary with all fields
            row = {
                "station_id": station['station_id'],
                "external_id": station.get('external_id', ''),
                "name": station['name'],
                "short_name": station.get('short_name', ''),
                "lat": station['lat'],
                "lon": station['lon'],
                "rental_methods": rental_methods_str,
                "capacity": station['capacity'],
                "electric_bike_surcharge_waiver": station.get('electric_bike_surcharge_waiver', False),
                "is_charging": station.get('is_charging', False),
                "eightd_has_key_dispenser": station.get('eightd_has_key_dispenser', False),
                "has_kiosk": station.get('has_kiosk', False),
                "last_updated": last_updated
            }
            writer.writerow(row)

    print(f"✅ Data saved to {csv_file} successfully!")

    # 5. Optional: Keep the console output (same as before)
    for station in stations:
        print(f"Station: {station['name']}")
        print(f"  ID: {station['station_id']}")
        print(f"  Capacity: {station['capacity']} docks")
        print(f"  Location: ({station['lat']}, {station['lon']})")
        print(f"  Rental Methods: {', '.join(station.get('rental_methods', []))}")
        print("-" * 30)

else:
    print(f"Failed to retrieve data. Status code: {response.status_code}")

Found 240 stations.

✅ Data saved to stations.csv successfully!
Station: Clark / Ontario
  ID: 3
  Capacity: 19 docks
  Location: (45.510587509716636, -73.56684565544128)
  Rental Methods: KEY, CREDITCARD
------------------------------
Station: Métro Berri-UQAM (St-Denis / de Maisonneuve)
  ID: 15
  Capacity: 15 docks
  Location: (45.51425199763688, -73.56150217976847)
  Rental Methods: KEY, CREDITCARD
------------------------------
Station: Marché St-Jacques (Atateken)
  ID: 17
  Capacity: 27 docks
  Location: (45.520665992418884, -73.5639146839003)
  Rental Methods: KEY, CREDITCARD
------------------------------
Station: Métro Sherbrooke (de Rigaud / Berri)
  ID: 19
  Capacity: 27 docks
  Location: (45.51814312154928, -73.56800436973572)
  Rental Methods: KEY, CREDITCARD
------------------------------
Station: Notre-Dame / St-Gabriel
  ID: 24
  Capacity: 11 docks
  Location: (45.50711760282556, -73.55504930019379)
  Rental Methods: KEY, CREDITCARD
------------------------------
Stati

In [7]:
# print first 10 rows of the second csv file dataframe
if len(dfs) > 1:
    print("First 10 rows of the second dataframe:")
    print(dfs[1].head(10))

First 10 rows of the second dataframe:
                          STARTSTATIONNAME       STARTSTATIONARRONDISSEMENT  \
0  Métro Champ-de-Mars (Viger / Sanguinet)                      Ville-Marie   
1  Métro Place-d'Armes (Viger / St-Urbain)                      Ville-Marie   
2               Émile-Duployé / Sherbrooke            Le Plateau-Mont-Royal   
3                       Marmier / St-Denis      Rosemont - La Petite-Patrie   
4                       Marmier / St-Denis      Rosemont - La Petite-Patrie   
5              du Parc-Lafontaine / Rachel            Le Plateau-Mont-Royal   
6               Ste-Catherine / St-Clément  Mercier - Hochelaga-Maisonneuve   
7                       Marmier / St-Denis      Rosemont - La Petite-Patrie   
8                         Dorion / Ontario                      Ville-Marie   
9                         Dorion / Ontario                      Ville-Marie   

   STARTSTATIONLATITUDE  STARTSTATIONLONGITUDE  \
0             45.510253             -73.5